In [ ]:

!pip install transformers seqeval evaluate conllu accelerate datasets -q


In [ ]:
test_tokens, test_pos = load_ud_data("en_ewt-ud-dev.conllu")

In [42]:
import numpy as np
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)

from evaluate import load
from conllu import parse


In [ ]:
def load_ud_data(file_path):

    tokens_list = []
    pos_tags_list = []

    with open(file_path, "r", encoding="utf-8") as f:
        sentences = parse(f.read())

    for sentence in sentences:

        tokens = []
        pos_tags = []

        for token in sentence:
            tokens.append(token["form"])
            pos_tags.append(token["upos"])

        tokens_list.append(tokens)
        pos_tags_list.append(pos_tags)

    return tokens_list, pos_tags_list

In [ ]:
import os

# Download the Universal Dependencies English Web Treebank (EWT) dataset files
if not os.path.exists("en_ewt-ud-train.conllu"):
    !wget https://raw.githubusercontent.com/UniversalDependencies/UD_English-EWT/master/en_ewt-ud-train.conllu
if not os.path.exists("en_ewt-ud-dev.conllu"):
    !wget https://raw.githubusercontent.com/UniversalDependencies/UD_English-EWT/master/en_ewt-ud-dev.conllu

train_tokens, train_pos = load_ud_data("en_ewt-ud-train.conllu")
test_tokens, test_pos = load_ud_data("en_ewt-ud-dev.conllu")

# Use only first 4000 samples
train_tokens = train_tokens[:4000]
train_pos = train_pos[:4000]

test_tokens = test_tokens[:500]
test_pos = test_pos[:500]

print(train_tokens[0])
print(train_pos[0])

--2026-04-05 14:14:46--  https://raw.githubusercontent.com/UniversalDependencies/UD_English-EWT/master/en_ewt-ud-train.conllu
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 15029817 (14M) [text/plain]
Saving to: ‘en_ewt-ud-train.conllu’

en_ewt-ud-train.con 100%[===================>]  14.33M  --.-KB/s    in 0.07s   

2026-04-05 14:14:47 (194 MB/s) - ‘en_ewt-ud-train.conllu’ saved [15029817/15029817]

--2026-04-05 14:14:47--  https://raw.githubusercontent.com/UniversalDependencies/UD_English-EWT/master/en_ewt-ud-dev.conllu
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request s

In [ ]:
labels = sorted(list(set(tag for sent in train_pos for tag in sent)))

label2id = {l:i for i,l in enumerate(labels)}
id2label = {i:l for l,i in label2id.items()}

num_labels = len(labels)

print(labels)



['ADJ', 'ADP', 'ADV', 'AUX', 'CCONJ', 'DET', 'INTJ', 'NOUN', 'NUM', 'PART', 'PRON', 'PROPN', 'PUNCT', 'SCONJ', 'SYM', 'VERB', 'X', '_']


In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [ ]:
def tokenize_and_align(tokens, labels):

    tokenized = tokenizer(
        tokens,
        truncation=True,
        is_split_into_words=True
    )

    word_ids = tokenized.word_ids()

    label_ids = []
    prev = None

    for word_idx in word_ids:

        if word_idx is None:
            label_ids.append(-100)

        elif word_idx != prev:
            label_ids.append(label2id[labels[word_idx]])

        else:
            label_ids.append(-100)

        prev = word_idx

    tokenized["labels"] = label_ids

    return tokenized

In [44]:
train_encodings = [tokenize_and_align(t, l) for t,l in zip(train_tokens, train_pos)]
test_encodings = [tokenize_and_align(t, l) for t,l in zip(test_tokens, test_pos)]


In [ ]:
class TokenDataset(torch.utils.data.Dataset):

    def __init__(self, encodings):
        self.encodings = encodings

    def __getitem__(self, idx):

        item = {k: torch.tensor(v) for k,v in self.encodings[idx].items()}
        return item

    def __len__(self):
        return len(self.encodings)


train_dataset = TokenDataset(train_encodings)
test_dataset = TokenDataset(test_encodings)

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized be

In [ ]:
data_collator = DataCollatorForTokenClassification(tokenizer)

In [ ]:
def compute_metrics(p):

    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [id2label[p] for (p,l) in zip(pred,label) if l != -100]
        for pred,label in zip(predictions,labels)
    ]

    true_labels = [
        [id2label[l] for (p,l) in zip(pred,label) if l != -100]
        for pred,label in zip(predictions,labels)
    ]

    results = metric.compute(
        predictions=true_predictions,
        references=true_labels
    )

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [ ]:
training_args = TrainingArguments(

    output_dir="./pos_model",

    learning_rate=2e-5,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    num_train_epochs=3,

    weight_decay=0.01,

    logging_steps=100
)

In [ ]:

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=test_dataset,

    data_collator=data_collator,

    compute_metrics=compute_metrics
)

In [46]:
metric = load("seqeval")

In [ ]:

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=test_dataset,

    data_collator=data_collator,

    compute_metrics=compute_metrics
)

In [45]:
results = trainer.evaluate()
print(results)

Step,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
25,No log,1.994013,0.400934,0.262113,0.316991,0.423670


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


{'eval_loss': 1.9940129518508911, 'eval_precision': 0.40093438097260564, 'eval_recall': 0.2621130084686936, 'eval_f1': 0.3169912693082606, 'eval_accuracy': 0.42366955846173765}


In [ ]:
from evaluate import load

metric = load("seqeval")

In [ ]:

trainer.evaluate()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: ADP seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: DET seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: PROPN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: VERB seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NOUN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171:

{'eval_loss': 2.963310956954956,
 'eval_model_preparation_time': 0.0057,
 'eval_precision': 0.03322784810126582,
 'eval_recall': 0.029154518950437316,
 'eval_f1': 0.03105819714560378,
 'eval_accuracy': 0.037809141525314,
 'eval_runtime': 77.1152,
 'eval_samples_per_second': 6.484,
 'eval_steps_per_second': 0.415}

In [ ]:

sentence = "John works at Google in California"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

inputs = tokenizer(sentence, return_tensors="pt")

inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs)

predictions = torch.argmax(outputs.logits, dim=2)

tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0].cpu())

pred_labels = [id2label[p.item()] for p in predictions[0]]

for token, label in zip(tokens, pred_labels):
    print(token, label)

[CLS] X
john SYM
works SYM
at PART
google INTJ
in PROPN
california INTJ
[SEP] PART


In [ ]:
sentence = "im very good at coding "

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

inputs = tokenizer(sentence, return_tensors="pt")

inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs)

predictions = torch.argmax(outputs.logits, dim=2)

tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0].cpu())

pred_labels = [id2label[p.item()] for p in predictions[0]]

for token, label in zip(tokens, pred_labels):
    print(token, label)


[CLS] ADP
im CCONJ
very SCONJ
good PROPN
at NOUN
coding SCONJ
[SEP] PART
